# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline rule:

I rank items using a simple score based only on signals available before
the prediction moment. Higher scores receive a higher-priority action.

Reason codes:
- HIGH_PRIORITY: strong signal for immediate action.
- MEDIUM_PRIORITY: moderate signal requiring review.
- LOW_PRIORITY: weak signal; monitor rather than act.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import os
from google.colab import files

# Upload the dataset
uploaded = files.upload()

# Get the uploaded CSV filename automatically
filename = next(iter(uploaded))

# Load dataset
df = pd.read_csv(filename)

print("Dataset:")
display(df.head())

print("Shape:", df.shape)


# -----------------------------------
# 1. BASELINE SCORE
# -----------------------------------

df["score"] = (
    (df["impressions"] >= 1000).astype(int) * 2
    + (df["staleness_days"] >= 14).astype(int) * 2
    + (df["position"] >= 8).astype(int) * 1
)


# -----------------------------------
# 2. REASON CODE
# -----------------------------------

df["reason_code"] = "REVIEW"

df.loc[
    (df["impressions"] >= 1000) &
    (df["staleness_days"] >= 14),
    "reason_code"
] = "STALE_HIGH_VOLUME"


# -----------------------------------
# 3. ACTION LABEL
# -----------------------------------

df["action"] = "MONITOR"

df.loc[df["score"] >= 3, "action"] = "PRIORITIZE"
df.loc[df["score"] >= 4, "action"] = "REVIEW_NOW"


# -----------------------------------
# 4. RANK
# -----------------------------------

df = df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)


# -----------------------------------
# 5. CREATE OUTPUT FOLDER
# -----------------------------------

os.makedirs("work/outputs", exist_ok=True)


# -----------------------------------
# 6. BUILD RANKED QUEUE
# -----------------------------------

output = df[
    [
        "rank",
        "item_id",
        "score",
        "reason_code",
        "action"
    ]
]


# -----------------------------------
# 7. SAVE CSV
# -----------------------------------

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


# -----------------------------------
# 8. DISPLAY TOP 20
# -----------------------------------

print("Top 20 ranked actions:")
display(output.head(20))

print("\nCSV created successfully:")
print("work/outputs/baseline_action_score.csv")

Saving ml07_baseline_practice_dataset.csv to ml07_baseline_practice_dataset.csv
Dataset:


,item_id,impressions,clicks,position,staleness_days,ctr
0,item_001,537,227,13,14,0.4227
1,item_002,3892,117,11,25,0.0301
2,item_003,3307,50,6,26,0.0151
3,item_004,2250,279,14,12,0.1240
4,item_005,2221,444,6,27,0.1999


Shape: (30, 6)
Top 20 ranked actions:


,rank,item_id,score,reason_code,action
0,1,item_002,5,STALE_HIGH_VOLUME,REVIEW_NOW
1,2,item_011,5,STALE_HIGH_VOLUME,REVIEW_NOW
2,3,item_024,5,STALE_HIGH_VOLUME,REVIEW_NOW
3,4,item_020,5,STALE_HIGH_VOLUME,REVIEW_NOW
4,5,item_019,5,STALE_HIGH_VOLUME,REVIEW_NOW
5,6,item_015,5,STALE_HIGH_VOLUME,REVIEW_NOW
6,7,item_026,5,STALE_HIGH_VOLUME,REVIEW_NOW
7,8,item_025,5,STALE_HIGH_VOLUME,REVIEW_NOW
8,9,item_009,4,STALE_HIGH_VOLUME,REVIEW_NOW
9,10,item_022,4,STALE_HIGH_VOLUME,REVIEW_NOW



CSV created successfully:
work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

I reviewed the top 20 ranked items using the baseline score, reason code, and action label.

For each item:
- Action: the recommended action based on the baseline score.
- Reason code: the main signal that caused the item to be prioritized.
- Confidence note: the recommendation is directional decision-support, not a guaranteed outcome.
- What would make it wrong: the signal may be temporary, incomplete, or influenced by another factor.

The top-ranked items should be reviewed first because they have the strongest combination of the selected baseline signals. Lower-ranked items may still need attention if additional context is available.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

Some lower-ranked recommendations may be weak because their score is
driven by only one signal. These should be reviewed rather than treated
as certain decisions.

Leakage check:

I did not use future outcomes, future-window metrics, labels, or
post-decision/product flags when calculating the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.